# Phase 4C.2 Lab - Frontier Pricing Boundary

Mục tiêu: hiểu Frontier adapter dùng ChromaDB + OpenAI như opt-in boundary.

Default expected output: tests pass/skipped without OpenAI call.

Safety: Frontier smoke chỉ chạy khi có `ENABLE_REAL_MODEL_CALLS=true`,
`PRICER_CHROMADB_PATH`, `PRICER_FRONTIER_MODEL_ID`, và `OPENAI_API_KEY`.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy Frontier-related default tests

Command này kiểm tra missing config/dependency và fallback behavior.


In [ ]:
run(["uv", "run", "pytest", "tests/test_real_pricing.py", "tests/test_real_pricing_frontier.py", "-q", "--tb=short"], timeout=180)


## 2. Kiểm tra Frontier env readiness mà không in secrets

Expected default: các giá trị config thường chưa đủ, nên real smoke sẽ skip.


In [ ]:
required = {
    "ENABLE_REAL_MODEL_CALLS": os.getenv("ENABLE_REAL_MODEL_CALLS", ""),
    "PRICER_CHROMADB_PATH": os.getenv("PRICER_CHROMADB_PATH", ""),
    "PRICER_FRONTIER_MODEL_ID": os.getenv("PRICER_FRONTIER_MODEL_ID", ""),
    "OPENAI_API_KEY": "<set>" if os.getenv("OPENAI_API_KEY") else "",
}

for key, value in required.items():
    if key == "OPENAI_API_KEY":
        print(f"{key}={value}")
    elif key == "PRICER_CHROMADB_PATH" and value:
        print(f"{key}=<path set, exists={Path(value).exists()}>")
    else:
        print(f"{key}={value or '<unset>'}")


## 3. Opt-in Frontier smoke cell

Only runs when env is configured. This may call OpenAI and access local
ChromaDB.


In [ ]:
chromadb_path = os.getenv("PRICER_CHROMADB_PATH", "")
frontier_ready = (
    os.getenv("ENABLE_REAL_MODEL_CALLS", "").strip().lower() == "true"
    and chromadb_path
    and Path(chromadb_path).exists()
    and os.getenv("PRICER_FRONTIER_MODEL_ID")
    and os.getenv("OPENAI_API_KEY")
)
if frontier_ready:
    run(["uv", "run", "pytest", "tests/test_real_pricing_frontier.py", "-q", "--tb=short"], timeout=300)
else:
    print("Skipped Frontier smoke. Requires real model flag, ChromaDB path, model id, and OPENAI_API_KEY.")


## 4. Cách đọc kết quả

- Default skip is correct when secrets/config are absent.
- Frontier smoke pass means vectorstore + embedding + OpenAI call succeeded.
- Do not paste API keys or raw model payloads if smoke fails.
